In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "validation").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.notebook_bootstrap import bootstrap

bedrock_model_arn, load_workshop_state, persist_workshop_state_file = bootstrap()


# Processamento de dados multimodal - Exemplo completo (end to end) usando Amazon Bedrock Knowledge Bases para texto e imagens

O RAG multimodal pode analisar e aproveitar insights tanto de dados textuais quanto visuais, como imagens, gráficos, diagramas e tabelas. O Bedrock Knowledge Bases oferece um fluxo de trabalho gerenciado de Retrieval-Augmented Generation (RAG) de ponta a ponta que permite aos clientes criar aplicações de IA generativa altamente precisas, de baixa latência, seguras e customizadas, incorporando informações contextuais de suas próprias data sources.

O Bedrock Knowledge Bases extrai conteúdo tanto de dados textuais quanto visuais, gera embeddings semânticos usando o modelo de embedding selecionado e os armazena no vector store escolhido. Isso permite que os usuários recuperem e gerem respostas a perguntas derivadas não apenas de texto, mas também de dados visuais. Além disso, os resultados recuperados agora incluem atribuição de fonte para dados visuais, aumentando a transparência e construindo confiança nas respostas geradas.

Você pode escolher entre: Amazon Bedrock Data Automation, um serviço gerenciado que extrai automaticamente conteúdo de dados multimodais (atualmente em Preview), ou FMs como os foundation models atuais do Bedrock, com a flexibilidade de customizar o prompt padrão.

Este notebook fornece código de exemplo para construir um RAG Multimodal usando Amazon Bedrock Knowledge Bases.

#### Etapas: 
- Criar a execution role da Knowledge Base com as policies necessárias para acessar/gravar dados de/para o S3 e os Foundation models necessários.
- Criar uma knowledge base com documentos de conteúdo rico
- Criar data source(s) dentro da knowledge base
- Iniciar jobs de ingestão usando as APIs da KB que irão ler dados da data source, fazer o parsing dos documentos (imagens, gráficos, tabelas etc.) usando Bedrock Data Automation ou Foundation model, fazer o chunking, converter os chunks em embeddings usando o modelo Amazon Titan Embeddings e então armazenar esses embeddings no AOSS. Tudo isso sem precisar construir, implantar e gerenciar o data pipeline.

Uma vez que os dados estejam disponíveis na Bedrock Knowledge Base, uma aplicação de perguntas e respostas pode ser construída usando as APIs da Knowledge Base fornecidas pelo Amazon Bedrock.


#### Pré-requisitos:

Certifique-se de habilitar o acesso aos modelos `us.anthropic.claude-haiku-4-5-20251001-v1:0` (modelo de geração de texto padrão, configurável via `BEDROCK_TEXT_MODEL_ID`), `Amazon Nova Micro` e `Titan Text Embeddings V2` no console do Amazon Bedrock

<div class="alert alert-block alert-info">
<b>Nota:</b> Por favor, execute o notebook célula por célula em vez de usar a opção "Run All Cells".
</div>

### 0 - Setup
Antes de executar o restante deste notebook, você precisará executar as células abaixo para (garantir que as bibliotecas necessárias estejam instaladas e) conectar-se ao Bedrock.

Por favor, ignore qualquer erro de dependência do pip (se você ver algum durante a instalação das bibliotecas)

In [ ]:
# %pip install --upgrade pip --quiet
# %pip install -r ../../requirements.txt --no-deps --quiet
# %pip install -r ../../requirements.txt --upgrade --quiet
%pip install pillow --quiet

In [ ]:
# %pip install --upgrade boto3
import boto3
print(boto3.__version__)

In [ ]:
# Kernel restart is intentionally skipped in corrected notebooks.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import sys
import time
import boto3
import logging
import pprint
import json

# Set the path to import module
from pathlib import Path
current_path = Path().resolve()
current_path = current_path.parent.parent
if str(current_path) not in sys.path:
    sys.path.append(str(current_path))
# Print sys.path to verify
# print(sys.path)

from utils.knowledge_base import BedrockKnowledgeBase

In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

In [ ]:
import uuid

suffix = uuid.uuid4().hex[:12]
knowledge_base_name = f"bedrock-multi-modal-kb-{suffix}"
knowledge_base_description = "Multi-modal RAG knowledge base."

bucket_name = f'{knowledge_base_name}'
# intermediate_bucket_name = f'{knowledge_base_name}-mm-storage'
foundation_model = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")


#### Você pode adicionar múltiplas data sources (S3, Sharepoint) a uma Knowledge Base multimodal. Para este notebook, testaremos a criação da Knowledge Base com um S3 Bucket.


Cada data source pode ter pré-requisitos diferentes, consulte a documentação da AWS para mais informações.

In [ ]:
## Please uncomment the data sources that you want to add and update the placeholder values accordingly.

data_sources=[
                {"type": "S3", "bucket_name": bucket_name}, 

                # {"type": "SHAREPOINT", "tenantId": "888d0b57-69f1-4fb8-957f-e1f0bedf64de", "domain": "yourdomain",
                #   "authType": "OAUTH2_CLIENT_CREDENTIALS",
                #  "credentialsSecretArn": f"arn:aws::secretsmanager:{region_name}:secret:<<your_secret_name>>",
                #  "siteUrls": ["https://yourdomain.sharepoint.com/sites/mysite"]
                # },
            ]
                
pp = pprint.PrettyPrinter(indent=2)

### 1 - Criar Knowledge Base com multimodalidade

In [ ]:
# For multi-modal RAG While instantiating BedrockKnowledgeBase, pass multi_modal= True and choose the parser you want to use

knowledge_base = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name}',
    kb_description=knowledge_base_description,
    data_sources=data_sources,
    multi_modal= True,
    parser='BEDROCK_FOUNDATION_MODEL', # BEDROCK_DATA_AUTOMATION
    chunking_strategy = "FIXED_SIZE", 
    suffix = f'{suffix}-f'
)

# Keep all S3 resources created for this notebook discoverable by the workshop cleanup.
for created_bucket in {
    bucket_name,
    getattr(knowledge_base, "intermediate_bucket_name", None),
}:
    if created_bucket:
        s3_client.put_bucket_tagging(
            Bucket=created_bucket,
            Tagging={"TagSet": [{"Key": "workshop-kb", "Value": "true"}, {"Key": "rag-workshop", "Value": "true"}]},
        )


### 2 - Ingestão de dados
Vamos baixar um PDF de conteúdo rico disponível publicamente e fazer o upload para um bucket S3

In [ ]:
import os

def create_directory(directory_name):    
    if not os.path.exists(directory_name):
        os.makedirs(directory_name)
        print(f"Directory '{directory_name}' created successfully.")
    else:
        print(f"Directory '{directory_name}' already exists.")

# Call the function to create the directory
create_directory("mm-data")

In [ ]:
import os
from pathlib import Path
from time import sleep
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import shutil


def download_file(url, filename, attempts=3, timeout=60):
    destination = Path(filename)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".part")
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(request, timeout=timeout) as response:
                status = getattr(response, "status", 200)
                if status >= 400:
                    raise RuntimeError(f"HTTP {status} while downloading {url}")
                with temporary.open("wb") as output:
                    while True:
                        chunk = response.read(1024 * 1024)
                        if not chunk:
                            break
                        output.write(chunk)
            if temporary.stat().st_size < 1024:
                raise RuntimeError("Downloaded file is unexpectedly small")
            temporary.replace(destination)
            print(f"File downloaded successfully: {destination}")
            return destination
        except (HTTPError, URLError, TimeoutError, OSError, RuntimeError) as error:
            last_error = error
            temporary.unlink(missing_ok=True)
            if attempt < attempts:
                sleep(2 ** (attempt - 1))

    raise RuntimeError(f"Could not download {url} after {attempts} attempts") from last_error


filename = "./mm-data/tornadoes_report.pdf"
url = "https://www.congress.gov/crs_external_products/IF/PDF/IF12695/IF12695.3.pdf"
download_file(url, filename)


##### Upload de dados para o bucket S3 da data source

In [ ]:
def upload_directory(path, bucket_name):
        for root,dirs,files in os.walk(path):
            for file in files:
                file_to_upload = os.path.join(root,file)
                print(f"uploading file {file_to_upload} to {bucket_name}")
                s3_client.upload_file(file_to_upload,bucket_name,file)

upload_directory("./mm-data", bucket_name)

### Iniciar job de ingestão
Uma vez que a KB e a(s) data source(s) foram criadas, podemos iniciar o job de ingestão para cada data source.
Durante o job de ingestão, a KB irá buscar os documentos da data source, fazer o parsing do documento para extrair texto, fazer o chunking com base no tamanho de chunking fornecido, criar embeddings de cada chunk e então gravá-los no banco de dados vetorial, neste caso o OSS.

NOTA: Atualmente, você só pode iniciar um job de ingestão por vez.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base.start_ingestion_job()

In [ ]:
# keep the kb_id for invocation later in the invoke request
kb_id = knowledge_base.get_knowledge_base_id()
print("Notebook state is persisted by the sequential runner.")

### 4 - Testar a Knowledge Base
Agora que a Knowledge Base está disponível, podemos testá-la usando as funções [**retrieve**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve.html) e [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html). 

#### Testando a Knowledge Base com a API Retrieve and Generate

Vamos primeiro testar a knowledge base usando a API retrieve and generate. Com essa API, o Bedrock cuida de recuperar as referências necessárias da knowledge base e gerar a resposta final usando um foundation model do Bedrock.

query = `Summarize annual trends of tornado reports and how it varies year over year.`

A resposta correta para essa query deve ser obtida a partir de um gráfico/chart do documento PDF.

In [ ]:
query = "Summarize annual trends of tornado reports and how it varies year over year."

In [ ]:
foundation_model = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")
# foundation_model = "amazon.nova-micro-v1:0"

response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

In [ ]:
from io import BytesIO
from urllib.parse import urlparse
from PIL import Image

s3_client_images = boto3.client('s3', region_name=region)

## Function to print retrieved response

def print_response(response):
#structure 'retrievalResults': list of contents. Each list has ['ResponseMetadata', 'citations', 'output', 'sessionId']
    print( f'OUTPUT: {response["output"]["text"]} \n')
    
    print(f'CITATION DETAILS: \n')
    
    for num, chunk in enumerate(response.get('citations', [])):
        print(f'CHUNK {num}',end='\n'*1)
        print("========")
        print(f'\t Generated  Response Text: ')
        print(f'\t ------------------------- ')
        generated = chunk.get('generatedResponsePart', {}).get('textResponsePart', {}).get('text', '')
        print(f'\t Generated  Response Text: ', generated, end='\n'*2)
        for i, ref in enumerate(chunk.get('retrievedReferences', [])):
            print(f'\t Retrieved References: ')
            print(f'\t ---------------------', )
            print(f'\n\t\t --> Location:', ref.get('location'))
            metadata = ref.get('metadata', {})
            print(f'\t\n\t\t --> Metadata: \n\t\t\t ---> Source', metadata.get('x-amz-bedrock-kb-source-uri', 'unavailable'))
            # print(f'\t\n\t\t\n\t\t\t ---> x-amz-bedrock-kb-description', ref['metadata']['x-amz-bedrock-kb-description'])
            image_uri = metadata.get('x-amz-bedrock-kb-byte-content-source')
            print(f'\t\n\t\t\n\t\t\t ---> x-amz-bedrock-kb-byte-content-source', image_uri or 'unavailable')
            print("")
            if not image_uri:
                print('\tNo image byte content attached to this reference.')
                continue
            parsed_image_uri = urlparse(image_uri)
            if parsed_image_uri.scheme != 's3' or not parsed_image_uri.netloc:
                print(f'Unable to display non-S3 image URI: {image_uri}')
                continue
            image_response = s3_client_images.get_object(
                Bucket=parsed_image_uri.netloc,
                Key=parsed_image_uri.path.lstrip('/'),
            )
            with BytesIO(image_response['Body'].read()) as image_buffer:
                display(Image.open(image_buffer).resize((400, 400)))

In [ ]:
print_response(response)

#### Testando a Knowledge Base com a API Retrieve
Se você precisa de uma camada extra de controle, pode recuperar os chunks que melhor correspondem à sua query usando a API retrieve. Nesta configuração, podemos definir o número desejado de resultados e controlar a resposta final com a lógica da sua própria aplicação. A API então fornece o conteúdo correspondente, sua localização no S3, o similarity score e o metadata do chunk.

In [ ]:
response_ret = bedrock_agent_runtime_client.retrieve(
    knowledgeBaseId=kb_id, 
    nextToken='string',
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults":5,
        } 
    },
    retrievalQuery={
        "text": "How many new positions were opened across Amazon's fulfillment and delivery network?"
    }
)

def response_print(retrieve_resp):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(retrieve_resp.get('retrievalResults', []),1):
        if 'text' in chunk.get('content', {}):
            print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        if 'byteContent' in chunk.get('content', {}):
            print(f'Chunk {num}: ',chunk['content']['byteContent'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk.get('location'),end='\n'*2)
        print(f'Chunk {num} Score: ',chunk.get('score'),end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk.get('metadata', {}),end='\n'*2)
        print("--------------------------------")

response_print(response_ret)

### Limpeza de recursos
Certifique-se de descomentar e executar a seção abaixo para excluir todos os recursos.

In [ ]:
# delete local directory
import shutil

dir_path = "mm-data" # Replace with the actual path

try:
    shutil.rmtree(dir_path)
    print(f"Directory '{dir_path}' and its contents have been deleted successfully.")
except FileNotFoundError:
    print(f"Directory '{dir_path}' not found.")
except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")
